In [28]:
# ⚙️ 1. Install Required Libraries
# %pip install pymupdf tqdm beautifulsoup4 pandas pyarrow


In [29]:
# 🧩 2. Fast Multi-Threaded PDF → Text Conversion
import os
import fitz  # PyMuPDF
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import re


BASE_DIR = "D:\LPA_MTech_Project\My_Datasets\SC_2010-2019"
OUT_DIR = "D:\LPA_MTech_Project\Extracted_Texts\Texts_2010-2019"
os.makedirs(OUT_DIR, exist_ok=True)

In [30]:
# === Text Cleaning Function ===
def clean_case_text(text):
    # 1️⃣ Remove repeated alphabet ladders (A–H blocks)
    text = re.sub(r'(\b[A-H]\b[\s\n]*){3,}', ' ', text)

    # 2️⃣ Remove numeric-only lines (page numbers)
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)

    # 3️⃣ Remove known SC headers/footers (but keep legal terms inside text)
    remove_patterns = [
        r"^\s*SUPREME COURT REPORTS\s*$",
        r"^\s*\[\d{4}\]\s*\d+\s*S\.C\.R\.\s*$",   # e.g. [2020] 1 S.C.R.
        r"^\s*REPORTABLE\s*$",
        r"^\s*NON-REPORTABLE\s*$",
        r"^\s*JUDGMENT\s*$",
        r"^\s*ORDER\s*$",
        r"^\s*CORAM\s*:.*$",                     # judge listing line
        r"^\s*BENCH\s*:.*$",                     # bench composition
    ]
    for p in remove_patterns:
        text = re.sub(p, " ", text, flags=re.IGNORECASE | re.MULTILINE)

    # 4️⃣ Remove dotted or dashed separators (---, …, etc.)
    text = re.sub(r"[-]{2,}|[.]{3,}|[_]{2,}", " ", text)

    # 5️⃣ Remove stray page titles like "SUPREME COURT REPORTS [2020] 1 S.C.R."
    text = re.sub(r"SUPREME COURT REPORTS\s*\[\d{4}\]\s*\d+\s*S\.C\.R\.", " ", text, flags=re.IGNORECASE)

    # 6️⃣ Normalize whitespace
    text = re.sub(r'\s{2,}', ' ', text)
    text = re.sub(r'\n{2,}', '\n', text)

    return text.strip()

In [31]:
# === PDF → Text Extraction Function ===
def extract_text_fitz(pdf_path, txt_path):
    try:
        with fitz.open(pdf_path) as doc:
            text = "".join([page.get_text("text") for page in doc])
        text = clean_case_text(text)
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(text)
    except Exception as e:
        print(f"⚠️ Error extracting {pdf_path}: {e}")

In [32]:
# === Collect All PDFs Across Years ===
pdf_files = []
for year in range(2010, 2020):
    year_path = os.path.join(BASE_DIR, str(year), "english")
    if os.path.exists(year_path):
        pdfs = [os.path.join(year_path, f) for f in os.listdir(year_path) if f.endswith(".pdf")]
        pdf_files.extend(pdfs)
        print(f"✅ {len(pdfs)} PDFs found in {year_path}")

print(f"\n📄 Total PDFs found: {len(pdf_files)}")

✅ 907 PDFs found in D:\LPA_MTech_Project\My_Datasets\SC_2010-2019\2010\english
✅ 848 PDFs found in D:\LPA_MTech_Project\My_Datasets\SC_2010-2019\2011\english
✅ 623 PDFs found in D:\LPA_MTech_Project\My_Datasets\SC_2010-2019\2012\english
✅ 853 PDFs found in D:\LPA_MTech_Project\My_Datasets\SC_2010-2019\2013\english
✅ 795 PDFs found in D:\LPA_MTech_Project\My_Datasets\SC_2010-2019\2014\english
✅ 760 PDFs found in D:\LPA_MTech_Project\My_Datasets\SC_2010-2019\2015\english
✅ 589 PDFs found in D:\LPA_MTech_Project\My_Datasets\SC_2010-2019\2016\english
✅ 732 PDFs found in D:\LPA_MTech_Project\My_Datasets\SC_2010-2019\2017\english
✅ 795 PDFs found in D:\LPA_MTech_Project\My_Datasets\SC_2010-2019\2018\english
✅ 1050 PDFs found in D:\LPA_MTech_Project\My_Datasets\SC_2010-2019\2019\english

📄 Total PDFs found: 7952


In [33]:
# === Parallel Extraction ===
with ThreadPoolExecutor(max_workers=6) as executor:
    list(tqdm(
        executor.map(
            lambda p: extract_text_fitz(
                p,
                os.path.join(OUT_DIR, os.path.basename(p).replace('.pdf', '.txt'))
            ),
            pdf_files
        ),
        total=len(pdf_files),
        desc="Extracting & Cleaning PDFs"
    ))

print(f"\n✅ All PDFs converted and cleaned → saved in: {OUT_DIR}")

Extracting & Cleaning PDFs: 100%|██████████| 7952/7952 [19:16<00:00,  6.87it/s]  


✅ All PDFs converted and cleaned → saved in: D:\LPA_MTech_Project\Extracted_Texts\Texts_2010-2019
